In [ ]:
dataset = []

with open('dataset/cat-facts.txt', 'r') as f:
    dataset = f.readlines()

print(f'Loaded {len(dataset)} entries.')

In [ ]:
# OpenAI client
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

client = OpenAI()

response = client.embeddings.create(
    model='text-embedding-3-small', 
    input='Testing embedding model'
)
embedding_vector = response.data[0].embedding

print(f'Vectgor dimensions: {len(embedding_vector)}')
print(f'sample values: {embedding_vector[:5]}')

In [ ]:
# Vector Database

VECTOR_DB = []

def add_chunk_to_database(chunk):
    response = client.embeddings.create(
        model='text-embedding-3-small',
        input=chunk
    )
    embedding_vector = response.data[0].embedding
    VECTOR_DB.append((chunk, embedding_vector))

for i, chunk in enumerate(dataset):
    add_chunk_to_database(chunk)
    print(f'Added chunk {i+1}/{len(dataset)} to the database.')

In [ ]:
import numpy as np

def cosine_similarity_numpy(v1, v2):
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    
    # Prevent division by zero
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0
        
    return dot_product / (norm_v1 * norm_v2)

# Example usage
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print(cosine_similarity_numpy(a, b))  # Output: 0.9746318461970762

def retrieve(query, top_n=3):
    response = client.embeddings.create(
        model='text-embedding-3-small',
        input=query
    )
    query_vector = response.data[0].embedding

    similarities = []

    for chunk, chunk_embedding in VECTOR_DB:
        similarity = cosine_similarity_numpy(query_vector, embedding_vector)
        similarities.append((chunk, similarity))

    similarities.sort(key=lambda x: x[1], reverse=True)

    return similarities[:top_n]

In [ ]:
# Generation phase

input_query = input('Ask me a question: ')
retrieved_knowledge = retrieve(input_query)

print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
    print(f'similarity: {similarity} \nchunk: {chunk}')

instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:
{'\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}
'''

response = client.responses.create(
    model="gpt-5-mini",
    input = [
        {'role': 'system', 'content': instruction_prompt},
        {'role': 'user', 'content': input_query}
    ]
)

print('Chatbot response:')
print(response.output_text)
